In [1]:
import torch
import torch.nn as nn

from src.gtsrb import NUM_CLASSES, get_dataloaders, save_predictions

In [2]:
train_loader, val_loader, test_loader = get_dataloaders(img_size=32, batch_size=128)

In [4]:
def train_model(optimizer_name):

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    model = nn.Sequential(

        nn.Conv2d(3, 32, kernel_size=3, padding=1),
        nn.BatchNorm2d(32),
        nn.ReLU(),
        nn.MaxPool2d(2),

        nn.Conv2d(32, 64, kernel_size=3, padding=1),
        nn.BatchNorm2d(64),
        nn.ReLU(),
        nn.MaxPool2d(2),

        nn.Flatten(),

        nn.Linear(64 * 8 * 8, 128),
        nn.ReLU(),

        nn.Linear(128, NUM_CLASSES)

    ).to(device)

    criterion = nn.CrossEntropyLoss()

    if optimizer_name == "SGD":
        optimizer = torch.optim.SGD(
            model.parameters(),
            lr=0.001
        )

    elif optimizer_name == "Momentum":
        optimizer = torch.optim.SGD(
            model.parameters(),
            lr=0.001,
            momentum=0.9
        )

    elif optimizer_name == "Adam":
        optimizer = torch.optim.Adam(
            model.parameters(),
            lr=0.001
        )

    epochs = 5

    losses = []
    accuracies = []

    for epoch in range(epochs):

        model.train()

        running_loss = 0.0

        for images, labels in train_loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)

            loss = criterion(outputs, labels)

            optimizer.zero_grad()

            loss.backward()

            optimizer.step()

            running_loss += loss.item()

        epoch_loss = running_loss

        losses.append(epoch_loss)

        model.eval()

        correct = 0
        total = 0

        with torch.no_grad():

            for images, labels in val_loader:

                images = images.to(device)
                labels = labels.to(device)

                outputs = model(images)

                predictions = outputs.argmax(dim=1)

                correct += (
                    predictions == labels
                ).sum().item()

                total += labels.size(0)

        accuracy = 100 * correct / total

        accuracies.append(accuracy)

        print(
            f"{optimizer_name} | "
            f"Epoch {epoch+1} | "
            f"Loss: {epoch_loss:.4f} | "
            f"Validation Accuracy: {accuracy:.2f}%"
        )

    return model, losses, accuracies

In [5]:
def generate_predictions(model):

    device = torch.device(
        "cuda" if torch.cuda.is_available() else "cpu"
    )

    model.eval()

    predictions_list = []

    with torch.no_grad():

        for images, _ in test_loader:

            images = images.to(device)

            outputs = model(images)

            predictions = outputs.argmax(dim=1)

            predictions_list.extend(
                predictions.cpu().numpy()
            )

    return predictions_list

In [6]:
model_sgd, losses_sgd, accs_sgd = train_model("SGD")

c:\Users\rafae\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\utils\data\dataloader.py:1095: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


SGD | Epoch 1 | Loss: 599.2001 | Validation Accuracy: 18.36%
SGD | Epoch 2 | Loss: 550.1011 | Validation Accuracy: 23.74%
SGD | Epoch 3 | Loss: 513.4049 | Validation Accuracy: 28.90%
SGD | Epoch 4 | Loss: 480.1345 | Validation Accuracy: 33.58%
SGD | Epoch 5 | Loss: 448.7489 | Validation Accuracy: 37.71%


In [7]:
model_momentum, losses_momentum, accs_momentum = train_model("Momentum")

Momentum | Epoch 1 | Loss: 466.0285 | Validation Accuracy: 50.83%
Momentum | Epoch 2 | Loss: 262.8754 | Validation Accuracy: 72.02%
Momentum | Epoch 3 | Loss: 165.3658 | Validation Accuracy: 83.01%
Momentum | Epoch 4 | Loss: 112.4702 | Validation Accuracy: 88.93%
Momentum | Epoch 5 | Loss: 82.0353 | Validation Accuracy: 91.70%


In [8]:
model_adam, losses_adam, accs_adam = train_model("Adam")

Adam | Epoch 1 | Loss: 198.3147 | Validation Accuracy: 91.93%
Adam | Epoch 2 | Loss: 33.7381 | Validation Accuracy: 96.73%
Adam | Epoch 3 | Loss: 14.5002 | Validation Accuracy: 98.44%
Adam | Epoch 4 | Loss: 9.5110 | Validation Accuracy: 98.61%
Adam | Epoch 5 | Loss: 5.9212 | Validation Accuracy: 98.20%


In [10]:
y_pred_sgd = generate_predictions(model_sgd)

save_predictions(
    y_pred_sgd,
    "results/predicoes_batchnorm_sgd.csv",
    experiment_name="CNN BatchNorm SGD"
)

In [11]:
y_pred_momentum = generate_predictions(model_momentum)

save_predictions(
    y_pred_momentum,
    "results/predicoes_batchnorm_momentum.csv",
    experiment_name="CNN BatchNorm Momentum"
)

In [12]:
y_pred_adam = generate_predictions(model_adam)

save_predictions(
    y_pred_adam,
    "results/predicoes_batchnorm_adam.csv",
    experiment_name="CNN BatchNorm Adam"
)